In [1]:
import sys
import re

from pathlib import Path

import geopandas as gpd
from sqlalchemy import create_engine
from pyproj import CRS
import shapely.wkb as wkb
from shapely.geometry import box
from typing import Optional

from spatial_data_pipe import SpatialDataProcessor, ProcessingConfig, DatabaseConfig

# Your specific processing configuration
config = ProcessingConfig(
    target_crs=4326,  # WGS84 (same as before)
    bbox_filter=(-116.688, 32.64, -114.472, 34.034),  # Salton Trough bounds
    state_filter="CALIFORNIA",  # Filter to California only
    county_filter={"IMPERIAL", "RIVERSIDE", "SAN DIEGO"},  # Your specific counties
    preferred_layers=["California (CA)", "California"],  # For KMZ files
    verbose=True,  # Enable detailed logging
    drop_z_dimension=True  # Standardize to 2D geometries
)

# Your database configuration
db_config = DatabaseConfig(
    host="localhost",
    port=5432,
    user="postgres",
    password="1123",
    database="lithiumvalley",  # Your specific database
    schema="public",
    geom_column="geom"
)

# Create processor with your settings
processor = SpatialDataProcessor(
    config=config,
    output_dir="data/processed/surface",  # Your output directory
    #db_config=db_config
)
paths = [

    r"../../data/Surface/CalGem_Geothermal_Wells/CalGEM%3A_Geothermal_Wells.shp",  # Fixed: %3A instead of :
    r"../../data/SubSurface/GEOTHERM/GEOTHERM_ALL(GEO_ALL).csv"
]
gdfs = {}
all_wells = processor.process_file(    r"../../data/Surface/CalGem_Geothermal_Wells/CalGEM%3A_Geothermal_Wells.shp")
geo_wells = processor.process_file(r"../../data/SubSurface/GEOTHERM/GEOTHERM_ALL(GEO_ALL).csv")





Processing shapefile: ..\..\data\Surface\CalGem_Geothermal_Wells\CalGEM%3A_Geothermal_Wells.shp
Reading shapefile: ..\..\data\Surface\CalGem_Geothermal_Wells\CalGEM%3A_Geothermal_Wells.shp
Loaded 4336 features
Detected CRS: EPSG:3857
After bbox filtering: 1318 features
Returning gdf with 1318 features
Processing CSV: ..\..\data\SubSurface\GEOTHERM\GEOTHERM_ALL(GEO_ALL).csv
Reading CSV: ..\..\data\SubSurface\GEOTHERM\GEOTHERM_ALL(GEO_ALL).csv
Using columns: LATITUDE, LONGITUDE
Valid coordinates: 7494 out of 8082 rows
After bbox filtering: 327 features


In [2]:
import geopandas as gpd
import pandas as pd
import re

geo_wells = gpd.read_file(r"../../data/Surface/CalGem_Geothermal_Wells/CalGEM%3A_Geothermal_Wells.shp")
all_wells = pd.read_csv(r"../../data/SubSurface/GEOTHERM/GEOTHERM_ALL(GEO_ALL).csv")

def dms_to_dd(coord):
    match = re.match(r"(\d+)-([\d.]+)\s*([NSEW])", str(coord).strip())
    if not match:
        return None
    deg, minutes, hemi = match.groups()
    dd = float(deg) + float(minutes)/60
    if hemi in ['S','W']:
        dd = -dd
    return dd

all_wells['LAT_DD'] = all_wells['LATITUDE'].apply(dms_to_dd)
all_wells['LON_DD'] = all_wells['LONGITUDE'].apply(dms_to_dd)

all_wells_gdf = gpd.GeoDataFrame(
    all_wells,
    geometry=gpd.points_from_xy(all_wells['LON_DD'], all_wells['LAT_DD']),
    crs="EPSG:4326"
).to_crs(geo_wells.crs)



In [4]:
bbox = box(-116.688, 32.64, -114.472, 34.034)

# Make sure wells are in EPSG:4326 before filtering
all_wells_gdf = all_wells_gdf.to_crs("EPSG:4326")

# Filter wells that fall inside the bbox
subset = all_wells_gdf[all_wells_gdf.geometry.within(bbox)]
subset.shape

(327, 123)

In [40]:
subset.to_file("GEOTHERM_ALL.shp", driver="ESRI Shapefile", index=False)
subset.head()

,GRECORD_,NAME,KGRA,WELLSPRING,API,WARINGNO,COUNTRY,STATE,COUNTY,LATITUDE,...,C13CO2,TRITIUM,C14OFC02,OTHERADATA,QUALIFINFO,REFERENCE,RORGANIZAT,LAT_DD,LON_DD,geometry
510,76357,. MARTINEZ WELL,NaN,15S-16E-27N1,NaN,NaN,UNITED STATES,CALIFORNIA,IMPERIAL,32-48.68 N,...,NaN,NaN,NaN,NaN,REX REPORTED 34 DEGREES C. ON 1/20/72.,*CALIFORNIA DEPT. OF WATER RESOURCES,CALIF. DIVISION OF MINES AND GEOLOGY,32.811333,-115.313333,POINT (-115.31333 32.81133)
512,76385,A. AXLER NO. 2 WELL,NaN,14S-16E-16K1,NaN,NaN,UNITED STATES,CALIFORNIA,IMPERIAL,32-55.88 N,...,NaN,NaN,NaN,NaN,CO3 VALUE CORRECTED BY AUTHOR,"REED, 1975",CALIFORNIA DIVISION OF MINES AND GEOLOGY,32.931333,-115.319667,POINT (-115.31967 32.93133)
513,76345,A. BORCHARD WELL,NaN,14S-16E-4Q2,NaN,NaN,UNITED STATES,CALIFORNIA,IMPERIAL,32-57.48 N,...,NaN,NaN,NaN,NaN,CO3 VALUE CORRECTED BY AUTHOR,"REED, 1975",CALIFORNIA DIVISION OF MINES AND GEOLOGY,32.958000,-115.319500,POINT (-115.3195 32.958)
514,76333,A. FUSI JR. WELL,NaN,NaN,NaN,NaN,UNITED STATES,CALIFORNIA,IMPERIAL,32-48.68 N,...,NaN,NaN,NaN,NaN,CO3 VALUE CORRECTED BY AUTHOR,"REED, 1975",CALIFORNIA DIVISION OF MINES AND GEOLOGY,32.811333,-115.353833,POINT (-115.35383 32.81133)
515,76334,"A. FUSI, SR. WELL",NaN,15S-16E-29Q1,NaN,NaN,UNITED STATES,CALIFORNIA,IMPERIAL,32-48.68 N,...,NaN,NaN,NaN,NaN,CO3 VALUE CORRECTED BY AUTHOR,"REED, 1975",CALIFORNIA DIVISION OF MINES AND GEOLOGY,32.811333,-115.337000,POINT (-115.337 32.81133)


In [ ]:
all_wells_gdf.to_file("GEOTHERM_ALL.shp", driver="ESRI Shapefile", index=False)

In [ ]:
all_wells_gdf[['LATITUDE','LONGITUDE']]

In [ ]:
all_wells.columns
all_spatial_cols = ['Lat83','Long83']
for c in geo_wells.columns:
    occurences = geo_wells[c].value_counts()
    if len(occurences) > 2:
        print(c,occurences)










In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import re

# Assuming your well data is in a DataFrame called 'wells_df'
# If not, load it first

def parse_coordinates(lat_str, lon_str):
    """Parse coordinate strings like '33-58.13 N' to decimal degrees"""
    try:
        if pd.isna(lat_str) or pd.isna(lon_str):
            return np.nan, np.nan
        
        # Parse latitude (e.g., "33-58.13 N")
        lat_match = re.match(r'(\d+)-(\d+\.?\d*)\s*([NS])', str(lat_str))
        if lat_match:
            deg, min_sec, direction = lat_match.groups()
            lat_decimal = float(deg) + float(min_sec)/60
            if direction == 'S':
                lat_decimal = -lat_decimal
        else:
            lat_decimal = np.nan
            
        # Parse longitude (e.g., "116-29.75 W")
        lon_match = re.match(r'(\d+)-(\d+\.?\d*)\s*([EW])', str(lon_str))
        if lon_match:
            deg, min_sec, direction = lon_match.groups()
            lon_decimal = float(deg) + float(min_sec)/60
            if direction == 'W':
                lon_decimal = -lon_decimal
        else:
            lon_decimal = np.nan
            
        return lat_decimal, lon_decimal
    except:
        return np.nan, np.nan

def analyze_well_data(wells_df):
    """Comprehensive EDA for well data"""
    
    print("=== WELL DATA EXPLORATORY ANALYSIS ===\n")
    
    # Basic info
    print(f"Total wells: {len(wells_df)}")
    print(f"Columns: {wells_df.columns.tolist()}\n")
    
    # Parse coordinates
    wells_df[['lat_decimal', 'lon_decimal']] = wells_df.apply(
        lambda x: pd.Series(parse_coordinates(x['LATITUDE'], x['LONGITUDE'])), axis=1
    )
    
    # Lithium analysis
    li_data = wells_df['LI'].dropna()
    print(f"=== LITHIUM ANALYSIS ===")
    print(f"Wells with Li measurements: {len(li_data)}")
    print(f"Li range: {li_data.min():} - {li_data.max():} mg/L")
    print(f"Li mean: {li_data.mean():} mg/L")
    print(f"Li median: {li_data.median():} mg/L")
    print(f"High Li wells (>1 mg/L): {len(li_data[li_data > 1])}")
    print(f"Very high Li wells (>10 mg/L): {len(li_data[li_data > 10])}\n")
    
    # Geospatial analysis
    print(f"=== GEOSPATIAL ANALYSIS ===")
    print(f"County distribution:")
    print(wells_df['COUNTY'].value_counts())
    print(f"\nTownships (T): {wells_df['T'].value_counts().head(10).to_dict()}")
    print(f"Ranges (R): {wells_df['R'].value_counts().head(10).to_dict()}")
    print(f"Sections: {wells_df['SEC'].value_counts().head(10).to_dict()}\n")
    
    # Temperature analysis
    temp_data = wells_df['TEMP'].dropna()
    print(f"=== TEMPERATURE ANALYSIS ===")
    print(f"Wells with temperature: {len(temp_data)}")
    print(f"Temperature range: {temp_data.min():.1f} - {temp_data.max():.1f}°C")
    print(f"High temp wells (>100°C): {len(temp_data[temp_data > 100])}")
    print(f"Geothermal wells: {len(wells_df[wells_df['SAMPLEINFO'].str.contains('GEOTHERMAL', na=False)])}\n")
    
    # Geochemical correlations
    print(f"=== GEOCHEMICAL CORRELATIONS ===")
    geochem_cols = ['LI', 'NA', 'K', 'CL', 'CA', 'MG', 'SO4', 'HCO3', 'TDS', 'PH']
    available_cols = [col for col in geochem_cols if col in wells_df.columns]
    
    if len(available_cols) > 1:
        corr_data = wells_df[available_cols].corr()
        print("Correlation matrix (top correlations with Li):")
        if 'LI' in corr_data.columns:
            li_corr = corr_data['LI'].sort_values(ascending=False)
            print(li_corr.head(10))
    
    # Create visualizations
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # 1. Lithium distribution
    axes[0,0].hist(li_data, bins=20, alpha=0.7, color='green')
    axes[0,0].set_title('Lithium Distribution')
    axes[0,0].set_xlabel('Li (mg/L)')
    axes[0,0].set_ylabel('Count')
    
    # 2. Temperature distribution
    axes[0,1].hist(temp_data, bins=20, alpha=0.7, color='red')
    axes[0,1].set_title('Temperature Distribution')
    axes[0,1].set_xlabel('Temperature (°C)')
    axes[0,1].set_ylabel('Count')
    
    # 3. County distribution
    wells_df['COUNTY'].value_counts().plot(kind='bar', ax=axes[0,2], color='blue', alpha=0.7)
    axes[0,2].set_title('Wells by County')
    axes[0,2].tick_params(axis='x', rotation=45)
    
    # 4. Li vs Temperature scatter
    if len(li_data) > 0 and len(temp_data) > 0:
        common_idx = li_data.index.intersection(temp_data.index)
        if len(common_idx) > 0:
            axes[1,0].scatter(wells_df.loc[common_idx, 'TEMP'], 
                             wells_df.loc[common_idx, 'LI'], alpha=0.6)
            axes[1,0].set_xlabel('Temperature (°C)')
            axes[1,0].set_ylabel('Li (mg/L)')
            axes[1,0].set_title('Li vs Temperature')
    
    # 5. Li vs TDS scatter
    if 'TDS' in wells_df.columns:
        tds_data = wells_df['TDS'].dropna()
        common_idx = li_data.index.intersection(tds_data.index)
        if len(common_idx) > 0:
            axes[1,1].scatter(wells_df.loc[common_idx, 'TDS'], 
                             wells_df.loc[common_idx, 'LI'], alpha=0.6)
            axes[1,1].set_xlabel('TDS (mg/L)')
            axes[1,1].set_ylabel('Li (mg/L)')
            axes[1,1].set_title('Li vs TDS')
    
    # 6. Township distribution
    wells_df['T'].value_counts().head(15).plot(kind='bar', ax=axes[1,2], color='orange', alpha=0.7)
    axes[1,2].set_title('Wells by Township')
    axes[1,2].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # High-value targets summary
    print(f"=== HIGH-VALUE TARGETS ===")
    high_li = wells_df[wells_df['LI'] > 1]
    if len(high_li) > 0:
        print(f"High Li wells (>1 mg/L):")
        for _, well in high_li.iterrows():
            print(f"  {well['API']}: Li={well['LI']:.2f} mg/L, Temp={well['TEMP']:.1f}°C, County={well['COUNTY']}")
    
    return wells_df

# Run the analysis
wells_df = analyze_well_data(geo_wells)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

def analyze_chemistry_data(wells_df):
    """EDA focused on chemistry components and quantitative data"""
    
    print("=== CHEMISTRY & QUANTITATIVE DATA ANALYSIS ===\n")
    
    # Define chemistry columns (excluding coordinates, names, dates, etc.)
    chem_cols = ['LI', 'NA', 'K', 'CL', 'CA', 'MG', 'SO4', 'HCO3', 'CO3', 'TDS', 
                 'PH', 'TEMP', 'CONDUCTANC', 'AL', 'AS', 'B', 'MN', 'MO', 'NI', 
                 'NO3', 'PO4', 'SIO2', 'SR', 'V', 'ZN', 'DWATER', 'OWATER18', 'FLOW', 'DEPTH']
    
    # Filter to available chemistry columns
    available_chem = [col for col in chem_cols if col in wells_df.columns]
    print(f"Available chemistry parameters: {len(available_chem)}")
    print(f"Parameters: {available_chem}\n")
    
    # Data completeness analysis
    print("=== DATA COMPLETENESS ===")
    completeness = wells_df[available_chem].notna().sum().sort_values(ascending=False)
    for param, count in completeness.items():
        pct = (count / len(wells_df)) * 100
        print(f"{param:12}: {count:3d} wells ({pct:5.1f}%)")
    
    # Statistical summary of all chemistry parameters
    print(f"\n=== STATISTICAL SUMMARY ===")
    chem_stats = wells_df[available_chem].describe()
    print(chem_stats.round(3))
    
    # Lithium-specific analysis
    if 'LI' in available_chem:
        li_data = wells_df['LI'].dropna()
        print(f"\n=== LITHIUM DETAILED ANALYSIS ===")
        print(f"Total Li measurements: {len(li_data)}")
        print(f"Range: {li_data.min():.3f} - {li_data.max():.3f} mg/L")
        print(f"Mean ± Std: {li_data.mean():.3f} ± {li_data.std():.3f} mg/L")
        print(f"Median: {li_data.median():.3f} mg/L")
        print(f"Q25-Q75: {li_data.quantile(0.25):.3f} - {li_data.quantile(0.75):.3f} mg/L")
        
        # Li concentration categories
        li_categories = {
            'Very Low (<0.1 mg/L)': len(li_data[li_data < 0.1]),
            'Low (0.1-1.0 mg/L)': len(li_data[(li_data >= 0.1) & (li_data < 1.0)]),
            'Moderate (1.0-10 mg/L)': len(li_data[(li_data >= 1.0) & (li_data < 10)]),
            'High (10-100 mg/L)': len(li_data[(li_data >= 10) & (li_data < 100)]),
            'Very High (>100 mg/L)': len(li_data[li_data >= 100])
        }
        print(f"\nLi concentration categories:")
        for category, count in li_categories.items():
            print(f"  {category}: {count} wells")
    
    # Major ion analysis
    major_ions = ['NA', 'K', 'CL', 'CA', 'MG', 'SO4', 'HCO3', 'CO3']
    available_major = [ion for ion in major_ions if ion in available_chem]
    
    if len(available_major) > 0:
        print(f"\n=== MAJOR ION ANALYSIS ===")
        for ion in available_major:
            ion_data = wells_df[ion].dropna()
            if len(ion_data) > 0:
                print(f"{ion:4}: {len(ion_data):3d} wells, range: {ion_data.min():.1f} - {ion_data.max():.1f} mg/L")
    
    # Temperature and depth analysis
    if 'TEMP' in available_chem:
        temp_data = wells_df['TEMP'].dropna()
        print(f"\n=== TEMPERATURE ANALYSIS ===")
        print(f"Temperature measurements: {len(temp_data)}")
        print(f"Range: {temp_data.min():.1f} - {temp_data.max():.1f}°C")
        print(f"Mean: {temp_data.mean():.1f}°C")
        print(f"High temp wells (>100°C): {len(temp_data[temp_data > 100])}")
        print(f"Very high temp wells (>200°C): {len(temp_data[temp_data > 200])}")
    
    if 'DEPTH' in available_chem:
        depth_data = wells_df['DEPTH'].dropna()
        print(f"\n=== DEPTH ANALYSIS ===")
        print(f"Depth measurements: {len(depth_data)}")
        print(f"Range: {depth_data.min():.1f} - {depth_data.max():.1f} m")
        print(f"Mean: {depth_data.mean():.1f} m")
    
    # Geochemical correlations
    print(f"\n=== GEOCHEMICAL CORRELATIONS ===")
    # Focus on parameters with sufficient data (>50% completeness)
    sufficient_data = [col for col in available_chem if wells_df[col].notna().sum() > len(wells_df) * 0.5]
    
    if len(sufficient_data) > 1:
        corr_matrix = wells_df[sufficient_data].corr()
        
        # Top correlations with Li
        if 'LI' in sufficient_data:
            li_corr = corr_matrix['LI'].sort_values(ascending=False)
            print("Top correlations with Li:")
            for param, corr_val in li_corr.head(10).items():
                if param != 'LI':
                    print(f"  {param:8}: {corr_val:.3f}")
        
        # Strong correlations overall
        strong_corr = []
        for i in range(len(sufficient_data)):
            for j in range(i+1, len(sufficient_data)):
                param1, param2 = sufficient_data[i], sufficient_data[j]
                corr_val = corr_matrix.loc[param1, param2]
                if abs(corr_val) > 0.7:
                    strong_corr.append((param1, param2, corr_val))
        
        if strong_corr:
            print(f"\nStrong correlations (|r| > 0.7):")
            for param1, param2, corr_val in sorted(strong_corr, key=lambda x: abs(x[2]), reverse=True):
                print(f"  {param1:8} - {param2:8}: {corr_val:.3f}")
    
    # Create chemistry-focused visualizations
    fig, axes = plt.subplots(3, 3, figsize=(18, 15))
    fig.suptitle('Chemistry Data Analysis', fontsize=16)
    
    # 1. Li distribution
    if 'LI' in available_chem:
        li_data = wells_df['LI'].dropna()
        axes[0,0].hist(li_data, bins=20, alpha=0.7, color='green', edgecolor='black')
        axes[0,0].set_title('Lithium Distribution')
        axes[0,0].set_xlabel('Li (mg/L)')
        axes[0,0].set_ylabel('Count')
        axes[0,0].axvline(li_data.median(), color='red', linestyle='--', label=f'Median: {li_data.median():.2f}')
        axes[0,0].legend()
    
    # 2. Temperature distribution
    if 'TEMP' in available_chem:
        temp_data = wells_df['TEMP'].dropna()
        axes[0,1].hist(temp_data, bins=20, alpha=0.7, color='red', edgecolor='black')
        axes[0,1].set_title('Temperature Distribution')
        axes[0,1].set_xlabel('Temperature (°C)')
        axes[0,1].set_ylabel('Count')
    
    # 3. pH distribution
    if 'PH' in available_chem:
        ph_data = wells_df['PH'].dropna()
        axes[0,2].hist(ph_data, bins=20, alpha=0.7, color='blue', edgecolor='black')
        axes[0,2].set_title('pH Distribution')
        axes[0,2].set_xlabel('pH')
        axes[0,2].set_ylabel('Count')
    
    # 4. TDS distribution
    if 'TDS' in available_chem:
        tds_data = wells_df['TDS'].dropna()
        axes[1,0].hist(tds_data, bins=20, alpha=0.7, color='orange', edgecolor='black')
        axes[1,0].set_title('TDS Distribution')
        axes[1,0].set_xlabel('TDS (mg/L)')
        axes[1,0].set_ylabel('Count')
    
    # 5. Li vs Temperature
    if 'LI' in available_chem and 'TEMP' in available_chem:
        common_idx = wells_df['LI'].dropna().index.intersection(wells_df['TEMP'].dropna().index)
        if len(common_idx) > 0:
            axes[1,1].scatter(wells_df.loc[common_idx, 'TEMP'], 
                             wells_df.loc[common_idx, 'LI'], alpha=0.6, color='purple')
            axes[1,1].set_xlabel('Temperature (°C)')
            axes[1,1].set_ylabel('Li (mg/L)')
            axes[1,1].set_title('Li vs Temperature')
    
    # 6. Li vs TDS
    if 'LI' in available_chem and 'TDS' in available_chem:
        common_idx = wells_df['LI'].dropna().index.intersection(wells_df['TDS'].dropna().index)
        if len(common_idx) > 0:
            axes[1,2].scatter(wells_df.loc[common_idx, 'TDS'], 
                             wells_df.loc[common_idx, 'LI'], alpha=0.6, color='brown')
            axes[1,2].set_xlabel('TDS (mg/L)')
            axes[1,2].set_ylabel('Li (mg/L)')
            axes[1,2].set_title('Li vs TDS')
    
    # 7. Major ions boxplot
    if len(available_major) > 0:
        major_ion_data = wells_df[available_major].dropna()
        if len(major_ion_data) > 0:
            major_ion_data.boxplot(ax=axes[2,0], rot=45)
            axes[2,0].set_title('Major Ions Distribution')
            axes[2,0].set_ylabel('Concentration (mg/L)')
    
    # 8. Correlation heatmap
    if len(sufficient_data) > 1:
        corr_matrix = wells_df[sufficient_data].corr()
        im = axes[2,1].imshow(corr_matrix, cmap='RdBu_r', vmin=-1, vmax=1)
        axes[2,1].set_title('Correlation Heatmap')
        axes[2,1].set_xticks(range(len(sufficient_data)))
        axes[2,1].set_yticks(range(len(sufficient_data)))
        axes[2,1].set_xticklabels(sufficient_data, rotation=45, ha='right')
        axes[2,1].set_yticklabels(sufficient_data)
        plt.colorbar(im, ax=axes[2,1])
    
    # 9. Data completeness heatmap
    completeness_matrix = wells_df[available_chem].notna().astype(int)
    axes[2,2].imshow(completeness_matrix.T, cmap='YlOrRd', aspect='auto')
    axes[2,2].set_title('Data Completeness')
    axes[2,2].set_xlabel('Well Index')
    axes[2,2].set_ylabel('Parameter')
    axes[2,2].set_yticks(range(len(available_chem)))
    axes[2,2].set_yticklabels(available_chem)
    
    plt.tight_layout()
    plt.show()
    
    # Summary statistics for high-value wells
    print(f"\n=== HIGH-VALUE WELLS SUMMARY ===")
    if 'LI' in available_chem:
        high_li_wells = wells_df[wells_df['LI'] > 1].copy()
        if len(high_li_wells) > 0:
            print(f"High Li wells (>1 mg/L): {len(high_li_wells)}")
            summary_cols = ['LI', 'TEMP', 'TDS', 'PH', 'COUNTY']
            available_summary = [col for col in summary_cols if col in high_li_wells.columns]
            if available_summary:
                print(high_li_wells[available_summary].describe().round(3))
    
    return wells_df

# Run the analysis
wells_df = analyze_chemistry_data(geo_wells)

In [ ]:
def create_lithium_correlation_map(wells_df):
    """Create detailed lithium correlation analysis and visualization"""
    
    print("=== LITHIUM CORRELATION MAP ANALYSIS ===\n")
    # Define chemistry parameters to analyze
    chem_params = ['LI', 'NA', 'K', 'CL', 'CA', 'MG', 'SO4', 'HCO3', 'CO3', 'TDS', 
                   'PH', 'TEMP', 'CONDUCTANC', 'AL', 'AS', 'B', 'MN', 'MO', 'NI', 
                   'NO3', 'PO4', 'SIO2', 'SR', 'V', 'ZN', 'DWATER', 'OWATER18', 'FLOW', 'DEPTH']
    
    # Filter to available parameters with sufficient data
    available_params = [col for col in chem_params if col in wells_df.columns]
    sufficient_data = [col for col in available_params if wells_df[col].notna().sum() > len(wells_df) * 0.1]
    
    print(f"Parameters with sufficient data (>10%): {len(sufficient_data)}")
    print(f"Parameters: {sufficient_data}\n")
    
    if 'LI' not in sufficient_data:
        print("Lithium data not available or insufficient for correlation analysis")
        return
    
    # Calculate correlation matrix
    corr_matrix = wells_df[sufficient_data].corr()
    li_correlations = corr_matrix['LI'].sort_values(ascending=False)
    
    # Display all correlations with Li
    print("=== ALL CORRELATIONS WITH LITHIUM ===")
    for param, corr_val in li_correlations.items():
        if param != 'LI':
            significance = ""
            if abs(corr_val) > 0.7:
                significance = " (STRONG)"
            elif abs(corr_val) > 0.5:
                significance = " (MODERATE)"
            elif abs(corr_val) > 0.3:
                significance = " (WEAK)"
            print(f"{param:12}: {corr_val:7.3f}{significance}")
    
    # Create comprehensive correlation visualization
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    fig.suptitle('Lithium Correlation Analysis', fontsize=16, fontweight='bold')
    
    # 1. Correlation heatmap focused on Li
    li_corr_subset = corr_matrix.loc[sufficient_data, sufficient_data]
    im1 = axes[0,0].imshow(li_corr_subset, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    axes[0,0].set_title('Full Correlation Matrix', fontweight='bold')
    axes[0,0].set_xticks(range(len(sufficient_data)))
    axes[0,0].set_yticks(range(len(sufficient_data)))
    axes[0,0].set_xticklabels(sufficient_data, rotation=45, ha='right', fontsize=8)
    axes[0,0].set_yticklabels(sufficient_data, fontsize=8)
    
    # Highlight Li row and column
    li_idx = sufficient_data.index('LI')
    for i in range(len(sufficient_data)):
        axes[0,0].add_patch(plt.Rectangle((li_idx-0.5, i-0.5), 1, 1, fill=False, edgecolor='yellow', lw=2))
        axes[0,0].add_patch(plt.Rectangle((i-0.5, li_idx-0.5), 1, 1, fill=False, edgecolor='yellow', lw=2))
    
    plt.colorbar(im1, ax=axes[0,0], shrink=0.8)
    
    # 2. Li correlations bar chart
    li_corr_sorted = li_correlations.drop('LI').sort_values(key=abs, ascending=False)
    colors = ['red' if x < 0 else 'blue' for x in li_corr_sorted.values]
    bars = axes[0,1].bar(range(len(li_corr_sorted)), li_corr_sorted.values, color=colors, alpha=0.7)
    axes[0,1].set_title('Lithium Correlations (Ranked)', fontweight='bold')
    axes[0,1].set_xlabel('Parameters')
    axes[0,1].set_ylabel('Correlation Coefficient')
    axes[0,1].set_xticks(range(len(li_corr_sorted)))
    axes[0,1].set_xticklabels(li_corr_sorted.index, rotation=45, ha='right')
    axes[0,1].axhline(y=0, color='black', linestyle='-', alpha=0.3)
    axes[0,1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Moderate correlation')
    axes[0,1].axhline(y=-0.5, color='gray', linestyle='--', alpha=0.5)
    axes[0,1].axhline(y=0.7, color='orange', linestyle='--', alpha=0.7, label='Strong correlation')
    axes[0,1].axhline(y=-0.7, color='orange', linestyle='--', alpha=0.7)
    axes[0,1].legend()
    
    # 3. Top positive correlations scatter plots
    top_positive = li_corr_sorted[li_corr_sorted > 0.3].head(3)
    if len(top_positive) > 0:
        for i, (param, corr_val) in enumerate(top_positive.items()):
            if i < 3:
                common_idx = wells_df['LI'].dropna().index.intersection(wells_df[param].dropna().index)
                if len(common_idx) > 0:
                    axes[0,2].scatter(wells_df.loc[common_idx, param], 
                                     wells_df.loc[common_idx, 'LI'], 
                                     alpha=0.6, label=f'{param} (r={corr_val:.3f})')
        
        axes[0,2].set_xlabel('Parameter Value')
        axes[0,2].set_ylabel('Lithium (mg/L)')
        axes[0,2].set_title('Top Positive Correlations', fontweight='bold')
        axes[0,2].legend()
    
    # 4. Top negative correlations scatter plots
    top_negative = li_corr_sorted[li_corr_sorted < -0.3].head(3)
    if len(top_negative) > 0:
        for i, (param, corr_val) in enumerate(top_negative.items()):
            if i < 3:
                common_idx = wells_df['LI'].dropna().index.intersection(wells_df[param].dropna().index)
                if len(common_idx) > 0:
                    axes[1,0].scatter(wells_df.loc[common_idx, param], 
                                     wells_df.loc[common_idx, 'LI'], 
                                     alpha=0.6, label=f'{param} (r={corr_val:.3f})')
        
        axes[1,0].set_xlabel('Parameter Value')
        axes[1,0].set_ylabel('Lithium (mg/L)')
        axes[1,0].set_title('Top Negative Correlations', fontweight='bold')
        axes[1,0].legend()
    
    # 5. Correlation network visualization (simplified)
    strong_corr = []
    for param in sufficient_data:
        if param != 'LI':
            corr_val = corr_matrix.loc['LI', param]
            if abs(corr_val) > 0.3:
                strong_corr.append((param, corr_val))
    
    if strong_corr:
        # Create a simple network representation
        y_pos = np.arange(len(strong_corr))
        colors = ['red' if x[1] < 0 else 'blue' for x in strong_corr]
        bars = axes[1,1].barh(y_pos, [abs(x[1]) for x in strong_corr], color=colors, alpha=0.7)
        axes[1,1].set_yticks(y_pos)
        axes[1,1].set_yticklabels([x[0] for x in strong_corr])
        axes[1,1].set_xlabel('Absolute Correlation Strength')
        axes[1,1].set_title('Strong Correlations Network', fontweight='bold')
        
        # Add correlation values as text
        for i, (param, corr_val) in enumerate(strong_corr):
            axes[1,1].text(abs(corr_val) + 0.02, i, f'{corr_val:.3f}', 
                           va='center', fontweight='bold')
    
    # 6. Li vs Temperature colored by other parameters
    if 'TEMP' in sufficient_data:
        temp_li_idx = wells_df['LI'].dropna().index.intersection(wells_df['TEMP'].dropna().index)
        if len(temp_li_idx) > 0:
            scatter = axes[1,2].scatter(wells_df.loc[temp_li_idx, 'TEMP'], 
                                       wells_df.loc[temp_li_idx, 'LI'], 
                                       c=wells_df.loc[temp_li_idx, 'LI'], 
                                       cmap='viridis', alpha=0.7, s=50)
            axes[1,2].set_xlabel('Temperature (°C)')
            axes[1,2].set_ylabel('Lithium (mg/L)')
            axes[1,2].set_title('Li vs Temperature (colored by Li)', fontweight='bold')
            plt.colorbar(scatter, ax=axes[1,2], label='Li (mg/L)')
    
    plt.tight_layout()
    plt.show()
    
    # Statistical significance testing
    print(f"\n=== STATISTICAL SIGNIFICANCE TESTING ===")
    for param in sufficient_data:
        if param != 'LI':
            common_idx = wells_df['LI'].dropna().index.intersection(wells_df[param].dropna().index)
            if len(common_idx) > 10:  # Need sufficient data for testing
                li_values = wells_df.loc[common_idx, 'LI']
                param_values = wells_df.loc[common_idx, param]
                
                # Pearson correlation test
                corr_coef, p_value = stats.pearsonr(li_values, param_values)
                
                significance = ""
                if p_value < 0.001:
                    significance = " (***)"
                elif p_value < 0.01:
                    significance = " (**)"
                elif p_value < 0.05:
                    significance = " (*)"
                
                print(f"{param:12}: r={corr_coef:7.3f}, p={p_value:7.4f}{significance}")
    
    # Summary of key findings
    print(f"\n=== KEY CORRELATION FINDINGS ===")
    strong_positive = li_correlations.copy()[li_correlations > 0.7].drop('LI')
    
    if len(strong_positive) > 0:
        print(f"Strong positive correlations (r > 0.7):")
        for param, corr_val in strong_positive.items():
            print(f"  {param}: {corr_val:.3f}")
    

    return corr_matrix

# Run the lithium correlation analysis
create_lithium_correlation_map(geo_wells)
geo_wells.columns.tolist()

In [ ]:
# %%
def identify_high_lithium_indicators(wells_df):
    """Identify wells with high lithium indicators based on correlation analysis"""
    
    print("=== HIGH LITHIUM INDICATOR WELLS ANALYSIS ===\n")
    
    # Define high-value thresholds based on correlation analysis
    thresholds = {
        'LI': 1.0,        # High Li > 1 mg/L
        'TDS': 50000,     # High TDS > 50,000 mg/L (based on typical brine values)
        'TEMP': 100,      # High temperature > 100°C
        'CL': 30000,      # High chloride > 30,000 mg/L
        'NA': 20000,      # High sodium > 20,000 mg/L
        'K': 2000,        # High potassium > 2,000 mg/L
        'CA': 5000,       # High calcium > 5,000 mg/L
        'B': 100,         # High boron > 100 mg/L
        'SR': 500,        # High strontium > 500 mg/L
        'DEPTH': 1000     # Deep wells > 1,000 m
    }
    
    # Create indicator columns for each parameter
    indicator_cols = []
    for param, threshold in thresholds.items():
        if param in wells_df.columns:
            indicator_col_name = f'{param}_HIGH'
            wells_df[indicator_col_name] = wells_df[param] > threshold
            indicator_cols.append(indicator_col_name)
            print(f"Created {indicator_col_name} indicator (>{threshold})")
    
    # Calculate composite lithium potential score
    print(f"\n=== COMPOSITE LITHIUM POTENTIAL SCORING ===")
    
    # Weight factors based on correlation strength
    weights = {
        'TDS_HIGH': 0.25,      # r=0.987
        'CL_HIGH': 0.20,       # r=0.982
        'NA_HIGH': 0.20,       # r=0.997
        'CA_HIGH': 0.15,       # r=0.974
        'K_HIGH': 0.10,        # r=0.956
        'TEMP_HIGH': 0.10      # r=0.796
    }
    
    # Calculate weighted score
    wells_df['li_potential_score'] = 0
    for indicator, weight in weights.items():
        if indicator in wells_df.columns:
            wells_df['li_potential_score'] += wells_df[indicator] * weight
            print(f"Added {indicator} to score with weight {weight}")
    
    # Normalize score to 0-1 range
    max_score = sum(weights.values())
    wells_df['li_potential_score'] = wells_df['li_potential_score'] / max_score
    
    # Identify high-potential wells
    high_potential = wells_df[wells_df['li_potential_score'] >= 0.5].copy()
    very_high_potential = wells_df[wells_df['li_potential_score'] >= 0.8].copy()
    
    print(f"\nHigh potential wells (score >= 0.5): {len(high_potential)}")
    print(f"Very high potential wells (score >= 0.8): {len(very_high_potential)}")
    
    # Analyze high-potential wells
    if len(high_potential) > 0:
        print(f"\n=== HIGH POTENTIAL WELLS ANALYSIS ===")
        
        # Sort by potential score
        high_potential_sorted = high_potential.sort_values('li_potential_score', ascending=False)
        
        # Display top wells
        print(f"\nTop 10 High Potential Wells:")
        display_cols = ['API', 'COUNTY', 'LI', 'TEMP', 'TDS', 'CL', 'NA', 'DEPTH', 'li_potential_score']
        available_display = [col for col in display_cols if col in high_potential_sorted.columns]
        
        for i, (idx, well) in enumerate(high_potential_sorted.head(10).iterrows()):
            print(f"\n{i+1}. {well.get('API', 'Unknown')}")
            print(f"   County: {well.get('COUNTY', 'Unknown')}")
            print(f"   Li: {well.get('LI', 'N/A'):.2f} mg/L")
            print(f"   Temp: {well.get('TEMP', 'N/A'):.1f}°C")
            print(f"   TDS: {well.get('TDS', 'N/A'):,.0f} mg/L")
            print(f"   Cl: {well.get('CL', 'N/A'):,.0f} mg/L")
            print(f"   Na: {well.get('NA', 'N/A'):,.0f} mg/L")
            print(f"   Depth: {well.get('DEPTH', 'N/A'):.0f} m")
            print(f"   Potential Score: {well.get('li_potential_score', 'N/A'):.3f}")
        
        # Statistical summary of high-potential wells
        print(f"\n=== HIGH POTENTIAL WELLS STATISTICS ===")
        numeric_cols = ['LI', 'TEMP', 'TDS', 'CL', 'NA', 'K', 'CA', 'DEPTH']
        available_numeric = [col for col in numeric_cols if col in high_potential.columns]
        
        if available_numeric:
            print(high_potential[available_numeric].describe().round(3))
        
        # County distribution of high-potential wells
        if 'COUNTY' in high_potential.columns:
            print(f"\nCounty distribution of high-potential wells:")
            print(high_potential['COUNTY'].value_counts())
        
        # Township/Range analysis for high-potential wells
        if 'T' in high_potential.columns and 'R' in high_potential.columns:
            print(f"\nTop Townships with high potential:")
            township_counts = high_potential.groupby(['T', 'R']).size().sort_values(ascending=False)
            print(township_counts.head(10))
    
    # Create visualizations for high-potential wells
    if len(high_potential) > 0:
        print(f"\n=== CREATING HIGH POTENTIAL WELLS VISUALIZATIONS ===")
        
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        fig.suptitle('High Lithium Potential Wells Analysis', fontsize=16, fontweight='bold')
        
        # 1. Potential score distribution
        axes[0,0].hist(high_potential['li_potential_score'], bins=20, alpha=0.7, color='green', edgecolor='black')
        axes[0,0].set_title('Lithium Potential Score Distribution')
        axes[0,0].set_xlabel('Potential Score')
        axes[0,0].set_ylabel('Number of Wells')
        axes[0,0].axvline(0.8, color='red', linestyle='--', label='Very High Threshold')
        axes[0,0].legend()
        
        # 2. Li vs Potential Score
        if 'LI' in high_potential.columns:
            # Get common indices where both LI and li_potential_score are not null
            common_idx = high_potential['LI'].dropna().index.intersection(high_potential['li_potential_score'].dropna().index)
            if len(common_idx) > 0:
                axes[0,1].scatter(high_potential.loc[common_idx, 'li_potential_score'], 
                                high_potential.loc[common_idx, 'LI'], 
                                alpha=0.7, color='blue')
                axes[0,1].set_xlabel('Lithium Potential Score')
                axes[0,1].set_ylabel('Lithium (mg/L)')
                axes[0,1].set_title('Li vs Potential Score')
        
        # 3. Temperature vs Potential Score
        if 'TEMP' in high_potential.columns:
            temp_data = high_potential['TEMP'].dropna()
            if len(temp_data) > 0:
                axes[0,2].scatter(high_potential['li_potential_score'], temp_data, alpha=0.7, color='red')
                axes[0,2].set_xlabel('Lithium Potential Score')
                axes[0,2].set_ylabel('Temperature (°C)')
                axes[0,2].set_title('Temperature vs Potential Score')
        
        # 4. TDS vs Potential Score
        if 'TDS' in high_potential.columns:
            tds_data = high_potential['TDS'].dropna()
            if len(tds_data) > 0:
                axes[1,0].scatter(high_potential['li_potential_score'], tds_data, alpha=0.7, color='orange')
                axes[1,0].set_xlabel('Lithium Potential Score')
                axes[1,0].set_ylabel('TDS (mg/L)')
                axes[1,0].set_title('TDS vs Potential Score')
        
        # 5. County distribution of high-potential wells
        if 'COUNTY' in high_potential.columns:
            high_potential['COUNTY'].value_counts().plot(kind='bar', ax=axes[1,1], color='purple', alpha=0.7)
            axes[1,1].set_title('High Potential Wells by County')
            axes[1,1].tick_params(axis='x', rotation=45)
        
        # 6. Depth vs Potential Score
        if 'DEPTH' in high_potential.columns:
            depth_data = high_potential['DEPTH'].dropna()
            if len(depth_data) > 0:
                axes[1,2].scatter(high_potential['li_potential_score'], depth_data, alpha=0.7, color='brown')
                axes[1,2].set_xlabel('Lithium Potential Score')
                axes[1,2].set_ylabel('Depth (m)')
                axes[1,2].set_title('Depth vs Potential Score')
        
        plt.tight_layout()
        plt.show()
        
        # Create correlation matrix for high-potential wells only
        if len(high_potential) > 5:  # Need sufficient data
            print(f"\n=== CORRELATION ANALYSIS FOR HIGH POTENTIAL WELLS ===")
            
            # Select chemistry parameters
            chem_params = ['LI', 'TEMP', 'TDS', 'CL', 'NA', 'K', 'CA', 'MG', 'B', 'SR', 'DEPTH']
            available_chem = [col for col in chem_params if col in high_potential.columns]
            
            if len(available_chem) > 1:
                high_potential_corr = high_potential[available_chem].corr()
                
                # Focus on Li correlations
                if 'LI' in available_chem:
                    li_corr_high = high_potential_corr['LI'].sort_values(ascending=False)
                    print("Li correlations in high-potential wells:")
                    for param, corr_val in li_corr_high.items():
                        if param != 'LI':
                            print(f"  {param:8}: {corr_val:.3f}")
                
                # Create correlation heatmap
                plt.figure(figsize=(12, 10))
                im = plt.imshow(high_potential_corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
                plt.title('Correlation Matrix: High Potential Wells', fontsize=14, fontweight='bold')
                plt.xticks(range(len(available_chem)), available_chem, rotation=45, ha='right')
                plt.yticks(range(len(available_chem)), available_chem)
                plt.colorbar(im, label='Correlation Coefficient')
                plt.tight_layout()
                plt.show()
    
    # Export high-potential wells data
    if len(high_potential) > 0:
        print(f"\n=== EXPORTING HIGH POTENTIAL WELLS DATA ===")
        
        # Create summary dataframe
        export_cols = ['API', 'COUNTY', 'T', 'R', 'SEC', 'LI', 'TEMP', 'TDS', 'CL', 'NA', 'K', 'CA', 'B', 'SR', 'DEPTH', 'li_potential_score']
        available_export = [col for col in export_cols if col in high_potential.columns]
        
        export_data = high_potential[available_export].copy()
        export_data = export_data.sort_values('li_potential_score', ascending=False)
        
        # Save to CSV
        output_file = "high_lithium_potential_wells.csv"
        export_data.to_csv(output_file, index=False)
        print(f"Exported {len(export_data)} high-potential wells to {output_file}")
        
        # Display summary statistics
        print(f"\nExport Summary:")
        print(f"Total wells exported: {len(export_data)}")
        print(f"Columns exported: {available_export}")
        
        if 'LI' in export_data.columns:
            li_stats = export_data['LI'].describe()
            print(f"\nLithium statistics in exported wells:")
            print(f"  Mean: {li_stats['mean']:.3f} mg/L")
            print(f"  Median: {li_stats['50%']:.3f} mg/L")
            print(f"  Max: {li_stats['max']:.3f} mg/L")
            print(f"  Min: {li_stats['min']:.3f} mg/L")
    
    return high_potential, very_high_potential

# Run the high lithium indicators analysis
high_potential_wells, very_high_potential_wells = identify_high_lithium_indicators(geo_wells)


# %%
geo_wells.columns.tolist()


In [ ]:
# %%
def analyze_spatial_distribution_high_potential(wells_df, high_potential_df):
    """Analyze spatial distribution of high lithium potential wells"""
    
    print("=== SPATIAL DISTRIBUTION ANALYSIS OF HIGH POTENTIAL WELLS ===\n")
    
    # Check if we have coordinate data
    coord_cols = ['lat_decimal', 'lon_decimal', 'LATITUDE', 'LONGITUDE']
    available_coords = [col for col in coord_cols if col in high_potential_df.columns]
    
    if not available_coords:
        print("No coordinate data available for spatial analysis")
        return
    
    # Use available coordinate columns
    if 'lat_decimal' in high_potential_df.columns and 'lon_decimal' in high_potential_df.columns:
        lat_col, lon_col = 'lat_decimal', 'lon_decimal'
    elif 'LATITUDE' in high_potential_df.columns and 'LONGITUDE' in high_potential_df.columns:
        lat_col, lon_col = 'LATITUDE', 'LONGITUDE'
    else:
        print("Coordinate columns not found")
        return
    
    # Filter to wells with valid coordinates
    spatial_wells = high_potential_df.dropna(subset=[lat_col, lon_col]).copy()
    
    if len(spatial_wells) == 0:
        print("No wells with valid coordinates found")
        return
    
    print(f"Wells with spatial data: {len(spatial_wells)}")
    
    # Basic spatial statistics
    print(f"\n=== SPATIAL STATISTICS ===")
    print(f"Latitude range: {spatial_wells[lat_col].min():.4f} to {spatial_wells[lat_col].max():.4f}")
    print(f"Longitude range: {spatial_wells[lon_col].min():.4f} to {spatial_wells[lon_col].max():.4f}")
    
    # County-based spatial analysis
    if 'COUNTY' in spatial_wells.columns:
        print(f"\n=== COUNTY SPATIAL ANALYSIS ===")
        county_stats = spatial_wells.groupby('COUNTY').agg({
            lat_col: ['count', 'mean', 'std'],
            lon_col: ['mean', 'std'],
            'li_potential_score': ['mean', 'max'],
            'LI': ['mean', 'max']
        }).round(4)
        
        print(county_stats)
    
    # Township/Range spatial clustering
    if 'T' in spatial_wells.columns and 'R' in spatial_wells.columns:
        print(f"\n=== TOWNSHIP/RANGE SPATIAL CLUSTERING ===")
        
        # Group by Township and Range
        tr_groups = spatial_wells.groupby(['T', 'R']).agg({
            lat_col: ['count', 'mean'],
            lon_col: ['mean'],
            'li_potential_score': ['mean', 'max'],
            'LI': ['mean', 'max']
        }).round(4)
        
        # Sort by number of wells and average potential score
        tr_groups['well_count'] = tr_groups[(lat_col, 'count')]
        tr_groups['avg_potential'] = tr_groups[('li_potential_score', 'mean')]
        tr_groups['avg_li'] = tr_groups[('LI', 'mean')]
        
        # Top clusters by well count
        print("Top Township/Range clusters by number of wells:")
        top_by_count = tr_groups.sort_values('well_count', ascending=False).head(10)
        for (t, r), data in top_by_count.iterrows():
            print(f"  T{t}R{r}: {data['well_count']} wells, avg potential: {data['avg_potential']}, avg Li: {data['avg_li']}")
        
        # Top clusters by average potential score
        print(f"\nTop Township/Range clusters by average potential score:")
        top_by_potential = tr_groups.sort_values('avg_potential', ascending=False).head(10)
        for (t, r), data in top_by_potential.iterrows():
            print(f"  T{t}R{r}: avg potential: {data['avg_potential']}, {data['well_count']} wells, avg Li: {data['avg_li']}")
    
    # Create spatial visualizations
    try:
        import matplotlib.pyplot as plt
        
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle('Spatial Distribution of High Lithium Potential Wells', fontsize=16, fontweight='bold')
        
        # 1. Basic scatter plot
        scatter = axes[0,0].scatter(spatial_wells[lon_col], spatial_wells[lat_col], 
                                   c=spatial_wells['li_potential_score'], 
                                   cmap='viridis', s=50, alpha=0.7)
        axes[0,0].set_xlabel('Longitude')
        axes[0,0].set_ylabel('Latitude')
        axes[0,0].set_title('High Potential Wells - Colored by Potential Score')
        plt.colorbar(scatter, ax=axes[0,0], label='Lithium Potential Score')
        
        # 2. Scatter plot colored by Li concentration
        if 'LI' in spatial_wells.columns:
            li_scatter = axes[0,1].scatter(spatial_wells[lon_col], spatial_wells[lat_col], 
                                          c=spatial_wells['LI'], 
                                          cmap='plasma', s=50, alpha=0.7)
            axes[0,1].set_xlabel('Longitude')
            axes[0,1].set_ylabel('Latitude')
            axes[0,1].set_title('High Potential Wells - Colored by Li Concentration')
            plt.colorbar(li_scatter, ax=axes[0,1], label='Lithium (mg/L)')
        
        # 3. Scatter plot colored by temperature
        if 'TEMP' in spatial_wells.columns:
            temp_scatter = axes[1,0].scatter(spatial_wells[lon_col], spatial_wells[lat_col], 
                                            c=spatial_wells['TEMP'], 
                                            cmap='hot', s=50, alpha=0.7)
            axes[1,0].set_xlabel('Longitude')
            axes[1,0].set_ylabel('Latitude')
            axes[1,0].set_title('High Potential Wells - Colored by Temperature')
            plt.colorbar(temp_scatter, ax=axes[1,0], label='Temperature (°C)')
        
        # 4. Scatter plot colored by TDS
        if 'TDS' in spatial_wells.columns:
            tds_scatter = axes[1,1].scatter(spatial_wells[lon_col], spatial_wells[lat_col], 
                                           c=spatial_wells['TDS'], 
                                           cmap='YlOrRd', s=50, alpha=0.7)
            axes[1,1].set_xlabel('Longitude')
            axes[1,1].set_ylabel('Latitude')
            axes[1,1].set_title('High Potential Wells - Colored by TDS')
            plt.colorbar(tds_scatter, ax=axes[1,1], label='TDS (mg/L)')
        
        plt.tight_layout()
        plt.show()
        
    except ImportError:
        print("Matplotlib not available for spatial visualizations")
    
    # Identify spatial clusters/hotspots
    print(f"\n=== SPATIAL CLUSTER IDENTIFICATION ===")
    
    # Simple clustering based on coordinate proximity
    from sklearn.cluster import DBSCAN
    from sklearn.preprocessing import StandardScaler
    
    try:
        # Prepare coordinates for clustering
        coords = spatial_wells[[lat_col, lon_col]].values
        
        # Scale coordinates (important for DBSCAN)
        scaler = StandardScaler()
        coords_scaled = scaler.fit_transform(coords)
        
        # Apply DBSCAN clustering
        clustering = DBSCAN(eps=0.5, min_samples=3).fit(coords_scaled)
        
        # Add cluster labels
        spatial_wells['cluster'] = clustering.labels_
        
        # Analyze clusters
        n_clusters = len(set(clustering.labels_)) - (1 if -1 in clustering.labels_ else 0)
        n_noise = list(clustering.labels_).count(-1)
        
        print(f"Number of clusters: {n_clusters}")
        print(f"Number of noise points: {n_noise}")
        
        # Analyze each cluster
        if n_clusters > 0:
            print(f"\nCluster Analysis:")
            for cluster_id in range(n_clusters):
                cluster_wells = spatial_wells[spatial_wells['cluster'] == cluster_id]
                print(f"\nCluster {cluster_id}:")
                print(f"  Number of wells: {len(cluster_wells)}")
                print(f"  Average potential score: {cluster_wells['li_potential_score'].mean():.3f}")
                if 'LI' in cluster_wells.columns:
                    print(f"  Average Li: {cluster_wells['LI'].mean():.2f} mg/L")
                if 'TEMP' in cluster_wells.columns:
                    print(f"  Average temperature: {cluster_wells['TEMP'].mean():.1f}°C")
                if 'COUNTY' in cluster_wells.columns:
                    print(f"  Counties: {cluster_wells['COUNTY'].unique()}")
        
        # Export clustered data
        cluster_output = "high_potential_wells_with_clusters.csv"
        spatial_wells.to_csv(cluster_output, index=False)
        print(f"\nExported clustered data to {cluster_output}")
        
    except ImportError:
        print("Scikit-learn not available for clustering analysis")
    
    return spatial_wells

# Run spatial distribution analysis
if 'high_potential_wells' in locals() and len(high_potential_wells) > 0:
    spatial_high_potential = analyze_spatial_distribution_high_potential(geo_wells, high_potential_wells)
else:
    print("No high potential wells data available for spatial analysis")



In [ ]:
geo_wells.columns.tolist()

In [ ]:
# %%
def create_lithium_exploration_targets(wells_df, high_potential_df):
    """Create comprehensive lithium exploration targets based on all indicators"""
    
    print("=== LITHIUM EXPLORATION TARGETS CREATION ===\n")
    
    # Define target categories based on different criteria
    target_categories = {
        'High_Li_Concentration': {'LI': 10, 'description': 'Wells with Li > 10 mg/L'},
        'High_Temperature': {'TEMP': 150, 'description': 'Wells with temperature > 150°C'},
        'High_Salinity': {'TDS': 100000, 'description': 'Wells with TDS > 100,000 mg/L'},
        'Deep_Targets': {'DEPTH': 2000, 'description': 'Wells deeper than 2000m'},
        'High_Chloride': {'CL': 50000, 'description': 'Wells with Cl > 50,000 mg/L'},
        'High_Sodium': {'NA': 40000, 'description': 'Wells with Na > 40,000 mg/L'},
        'High_Potassium': {'K': 5000, 'description': 'Wells with K > 5,000 mg/L'},
        'High_Calcium': {'CA': 10000, 'description': 'Wells with Ca > 10,000 mg/L'},
        'High_Boron': {'B': 500, 'description': 'Wells with B > 500 mg/L'},
        'High_Strontium': {'SR': 1000, 'description': 'Wells with Sr > 1,000 mg/L'}
    }
    
    # Create target dataframes
    targets = {}
    target_summary = {}
    
    for target_name, criteria in target_categories.items():
        param = list(criteria.keys())[0]
        threshold = list(criteria.values())[0]
        description = criteria['description']
        
        if param in wells_df.columns:
            target_wells = wells_df[wells_df[param] > threshold].copy()
            targets[target_name] = target_wells
            target_summary[target_name] = {
                'count': len(target_wells),
                'threshold': threshold,
                'description': description,
                'param': param
            }
            
            print(f"{target_name}: {len(target_wells)} wells ({description})")
    
    # Create composite targets
    print(f"\n=== COMPOSITE TARGETS ===")
    
    # Multi-parameter targets
    composite_targets = {
        'Premium_Li_Targets': {
            'criteria': {
                'LI': 5,      # Li > 5 mg/L
                'TEMP': 120,  # Temp > 120°C
                'TDS': 75000  # TDS > 75,000 mg/L
            },
            'description': 'High Li + High Temp + High Salinity'
        },
        'Deep_Geothermal_Li': {
            'criteria': {
                'LI': 2,      # Li > 2 mg/L
                'TEMP': 100,  # Temp > 100°C
                'DEPTH': 1500 # Depth > 1500m
            },
            'description': 'Moderate Li + High Temp + Deep'
        },
        'High_Salinity_Li': {
            'criteria': {
                'LI': 1,      # Li > 1 mg/L
                'TDS': 100000, # TDS > 100,000 mg/L
                'CL': 50000   # Cl > 50,000 mg/L
            },
            'description': 'Any Li + Very High Salinity + High Cl'
        }
    }
    
    # Apply composite criteria
    for target_name, target_info in composite_targets.items():
        criteria = target_info['criteria']
        description = target_info['description']
        
        # Start with all wells
        composite_target = wells_df.copy()
        
        # Apply each criterion
        for param, threshold in criteria.items():
            if param in composite_target.columns:
                composite_target = composite_target[composite_target[param] > threshold]
        
        targets[target_name] = composite_target
        target_summary[target_name] = {
            'count': len(composite_target),
            'threshold': 'Composite',
            'description': description,
            'param': 'Multiple'
        }
        
        print(f"{target_name}: {len(composite_target)} wells ({description})")
    
    # Analyze target overlaps
    print(f"\n=== TARGET OVERLAP ANALYSIS ===")
    
    # Create overlap matrix
    target_names = list(targets.keys())
    overlap_matrix = pd.DataFrame(index=target_names, columns=target_names)
    
    for i, target1 in enumerate(target_names):
        for j, target2 in enumerate(target_names):
            if i == j:
                overlap_matrix.loc[target1, target2] = len(targets[target1])
            else:
                overlap = len(targets[target1].index.intersection(targets[target2].index))
                overlap_matrix.loc[target1, target2] = overlap
    
    print("Target overlap matrix (number of wells in common):")
    print(overlap_matrix)
    
    # Create target ranking
    print(f"\n=== TARGET RANKING BY POTENTIAL ===")
    
    target_ranking = []
    for target_name, target_data in targets.items():
        if len(target_data) > 0:
            # Calculate average potential score if available
            if 'li_potential_score' in target_data.columns:
                avg_potential = target_data['li_potential_score'].mean()
            else:
                avg_potential = 0
            
            # Calculate average Li concentration
            if 'LI' in target_data.columns:
                avg_li = target_data['LI'].mean()
            else:
                avg_li = 0
            
            # Calculate average temperature
            if 'TEMP' in target_data.columns:
                avg_temp = target_data['TEMP'].mean()
            else:
                avg_temp = 0
            
            target_ranking.append({
                'target_name': target_name,
                'well_count': len(target_data),
                'avg_potential_score': avg_potential,
                'avg_li': avg_li,
                'avg_temperature': avg_temp,
                'description': target_summary[target_name]['description']
            })
    
    # Sort by average potential score
    target_ranking.sort(key=lambda x: x['avg_potential_score'], reverse=True)
    
    print("Target ranking by average potential score:")
    for i, target in enumerate(target_ranking):
        print(f"{i+1}. {target['target_name']}")
        print(f"   Wells: {target['well_count']}")
        print(f"   Avg Potential Score: {target['avg_potential_score']:.3f}")
        print(f"   Avg Li: {target['avg_li']:.2f} mg/L")
        print(f"   Avg Temperature: {target['avg_temperature']:.1f}°C")
        print(f"   Description: {target['description']}")
        print()
    
    # Export target data
    print(f"=== EXPORTING TARGET DATA ===")
    
    for target_name, target_data in targets.items():
        if len(target_data) > 0:
            # Select key columns for export
            export_cols = ['API', 'COUNTY', 'T', 'R', 'SEC', 'LI', 'TEMP', 'TDS', 'CL', 'NA', 'K', 'CA', 'B', 'SR', 'DEPTH']
            if 'li_potential_score' in target_data.columns:
                export_cols.append('li_potential_score')
            
            available_export = [col for col in export_cols if col in target_data.columns]
            
            if available_export:
                export_filename = f"lithium_target_{target_name.lower().replace(' ', '_')}.csv"
                target_data[available_export].to_csv(export_filename, index=False)
                print(f"Exported {target_name}: {len(target_data)} wells to {export_filename}")
    
    # Create target summary report
    summary_report = pd.DataFrame(target_summary).T
    summary_report.to_csv("lithium_targets_summary.csv")
    print(f"Exported target summary to lithium_targets_summary.csv")
    
    return targets, target_summary, target_ranking

# Run the exploration targets creation
if 'high_potential_wells' in locals():
    exploration_targets, target_summary, target_ranking = create_lithium_exploration_targets(geo_wells, high_potential_wells)
else:
    print("No high potential wells data available for target creation")



In [ ]:
high_potential_wells['NAME']